# Rung 1 — Poisson baseline

**Goal of this notebook:** build the simplest defensible match predictor and *measure* it, so every later upgrade (Dixon–Coles, Elo priors, ML) can be judged against a real baseline.

The plan:
1. Load historical international results.
2. Fit the Poisson strength model (attack, defence, home advantage).
3. Sanity-check it on a few known matchups.
4. Backtest with RPS / Brier / log-loss.
5. Compare against the bookmaker and API-Football benchmarks.

The modelling code lives in `../src/` — open `poisson.py` alongside this; it's short and commented. The notebook is where you *drive* it and build intuition.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # so we can import src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data, poisson, evaluate

pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## 1. Load the data

Drop the Kaggle *International football results 1872–present* `results.csv` into `../data/raw/` and set `CSV` below.

Until then, `make_synthetic()` generates a fake dataset with known latent strengths so the whole pipeline runs end-to-end. Swap to the real CSV when you have it.

In [ ]:
CSV = "../data/raw/results.csv"

if os.path.exists(CSV):
    df = data.load_results(CSV)
    df = data.filter_recent(df, years=8)      # recent form matters more
    print(f"Loaded {len(df):,} real matches, {df['date'].min().date()} → {df['date'].max().date()}")
else:
    df = data.make_synthetic(n_teams=10, n_matches=600, seed=1)
    print(f"No CSV found — using {len(df):,} SYNTHETIC matches. Drop results.csv into data/raw/ to use real data.")

df.head()

## 2. Fit the Poisson model

`poisson.fit` reshapes the data to one row per (team, goals) and fits a Poisson GLM:

$$\log E[\text{goals}] = \text{intercept} + \beta_{\text{home}}\cdot\text{home} + \text{attack}_{\text{team}} + \text{defence}_{\text{opponent}}$$

The fitted coefficients *are* the team strengths. A positive `home` coefficient is the home-advantage effect (for neutral WC venues we'll later zero this out — see rung 3).

In [ ]:
model = poisson.fit(df, max_goals=10)
print(model.result.summary().tables[0])
print(f"\nHome-advantage coefficient (log scale): {model.result.params.get('home', float('nan')):.3f}")

### Inspect the learned strengths

Higher attack ⇒ scores more; **lower** defence coefficient ⇒ concedes fewer (it enters as the opponent's effect on your goals).

In [ ]:
params = model.result.params
attack = params[[i for i in params.index if i.startswith('C(team)')]].sort_values(ascending=False)
print("Strongest attacks (top 5):")
print(attack.head().rename(lambda s: s.replace('C(team)[T.','').rstrip(']')))

## 3. Sanity-check a fixture

Pick any two teams in the dataset and inspect: expected goals, the most-likely scorelines, and the win/draw/loss split. Do the numbers pass the smell test? (Stronger team favoured, totals in a sensible 0–4 range.)

In [ ]:
teams = model.teams
home_team, away_team = teams[0], teams[1]

lam_h, lam_a = model.expected_goals(home_team, away_team)
print(f"{home_team} vs {away_team}")
print(f"  expected goals: {lam_h:.2f} – {lam_a:.2f}")
print(f"  outcome probs : {model.outcome_probs(home_team, away_team)}")
print("  most-likely scores:")
for (i, j), p in model.most_likely_scores(home_team, away_team):
    print(f"    {i}-{j}: {p:.3f}")

In [ ]:
# Visualise the scoreline matrix
m = model.score_matrix(home_team, away_team)[:6, :6]
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(m, origin="lower", cmap="viridis")
ax.set_xlabel(f"{away_team} goals"); ax.set_ylabel(f"{home_team} goals")
ax.set_title("P(scoreline)"); fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## 4. Backtest with proper scoring

Train on an earlier slice, test on a later slice (never shuffle time). For each test match we produce `[home, draw, away]` probabilities and score them with **RPS** (the 1X2 standard), **Brier**, and **log-loss**. Lower is better.

A useful reference point: always-predict the base rates of the training set. Your model should beat that.

In [ ]:
split = df.sort_values("date").iloc[: int(len(df) * 0.8)], df.sort_values("date").iloc[int(len(df) * 0.8):]
train, test = split
m_bt = poisson.fit(train, max_goals=10)
known = set(m_bt.teams)

prob_rows, outcomes = [], []
for _, r in test.iterrows():
    if r.home_team not in known or r.away_team not in known:
        continue   # can't predict a team unseen in training
    p = m_bt.outcome_probs(r.home_team, r.away_team)
    prob_rows.append([p["home"], p["draw"], p["away"]])
    outcomes.append(evaluate.result_to_outcome(r.home_score, r.away_score))

model_scores = evaluate.mean_scores(prob_rows, outcomes)

# baseline: predict the training-set base rates for every match
base = train.assign(o=lambda d: [evaluate.result_to_outcome(h, a) for h, a in zip(d.home_score, d.away_score)])
rates = base["o"].value_counts(normalize=True).reindex(["home","draw","away"]).fillna(0).values
base_scores = evaluate.mean_scores([rates]*len(outcomes), outcomes)

print("Poisson model:", model_scores)
print("Base-rate    :", base_scores)
print("\n→ RPS improvement vs base rate:", round(base_scores['rps'] - model_scores['rps'], 4))

## 5. Compare against the real benchmarks (once you have the API key)

When the WC2026 fixtures are live, pull the two external benchmarks and score them the *same way*:

- **Bookmaker** — `api.odds(fixture_id)` → `evaluate.implied_probs_from_odds(...)` → RPS.
- **API-Football model** — `api.predictions(fixture_id)` → read its home/draw/away percentages → RPS.

If your Poisson model can't match the bookmaker on RPS, that's the signal to climb to rung 2 (Dixon–Coles). The cell below is a stub — it needs a live `fixture_id` and a key in `.env`.

In [ ]:
# Requires API_FOOTBALL_KEY in ../.env and a live fixture id.
# from src.api_client import ApiFootball
# api = ApiFootball()
# fx = api.fixtures()
# fixture_id = fx[0]["fixture"]["id"]
# print(api.predictions(fixture_id)[0]["predictions"])
# print(api.odds(fixture_id))
print("Benchmark comparison runs once you have a key + live fixtures. See API-Football Notes.md.")

## Next: rung 2 — Dixon–Coles

Two upgrades, both measurable against the RPS we just recorded:
1. **Low-score correction** — the independent-Poisson assumption under-counts 0-0/1-0/0-1; Dixon–Coles adds a correlation parameter τ for those cells.
2. **Time decay** — weight recent matches more (exp decay on match age), instead of the hard 8-year cutoff.

Implement in `src/` as `dixon_coles.py`, re-run the backtest, and only keep it if RPS drops.